# AgroClimate - Crop Climate Impact Prediction Model

This notebook demonstrates the AgroClimate machine learning model for predicting climate impact on crops.

## Objectives
- Predict climate risk scores for different crops
- Generate adaptive crop recommendations
- Analyze climate-crop relationships

## SDGs Addressed
- SDG 2: Zero Hunger
- SDG 13: Climate Action

In [ ]:
# Import necessary libraries
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_loader import DataLoader
from src.data_processing.preprocess import DataPreprocessor
from src.models.climate_risk_model import ClimateRiskModel
from src.models.crop_recommender import CropRecommender
from src.utils.visualization import AgroClimateVisualizer
from config.config import *

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

print("AgroClimate Analysis Notebook Initialized!")

## 1. Data Loading and Exploration

In [ ]:
# Load data
loader = DataLoader(DATA_RAW_PATH)

# Load climate and yield data
climate_df = loader.load_climate_data('climate_data.csv')
yield_df = loader.load_crop_yield_data('crop_yield_data.csv')

print("Climate Data Shape:", climate_df.shape)
print("Yield Data Shape:", yield_df.shape)

# Display first few rows
print("\nClimate Data Sample:")
display(climate_df.head())

print("\nYield Data Sample:")
display(yield_df.head())

In [ ]:
# Merge datasets
merged_df = loader.merge_climate_yield_data(climate_df, yield_df)
print("Merged Dataset Shape:", merged_df.shape)
print("\nMerged Data Sample:")
display(merged_df.head())

# Basic statistics
print("\nBasic Statistics:")
display(merged_df.describe())

## 2. Data Preprocessing

In [ ]:
# Preprocess data
preprocessor = DataPreprocessor()
processed_df = preprocessor.preprocess_data(merged_df)

print("Processed Dataset Shape:", processed_df.shape)
print("\nNew Features Created:")
new_features = [col for col in processed_df.columns if col not in merged_df.columns]
print(new_features)

# Check for missing values
print("\nMissing Values:")
print(processed_df.isnull().sum())

## 3. Exploratory Data Analysis

In [ ]:
# Initialize visualizer
visualizer = AgroClimateVisualizer()

# Generate climate trends
visualizer.plot_climate_trends(processed_df, save_plot=False)

In [ ]:
# Crop yield analysis
visualizer.plot_crop_yield_analysis(processed_df, save_plot=False)

In [ ]:
# Risk assessment visualization
visualizer.plot_risk_assessment(processed_df, save_plot=False)

## 4. Model Training

In [ ]:
# Initialize and train models
model = ClimateRiskModel()

# Prepare features
X, y_risk, y_yield = model.prepare_features(processed_df)

print("Feature Matrix Shape:", X.shape)
print("Features:", list(X.columns))
print("\nTarget Variables:")
print(f"Risk Score Range: {y_risk.min():.3f} - {y_risk.max():.3f}")
print(f"Yield Range: {y_yield.min():.2f} - {y_yield.max():.2f} tons/hectare")

In [ ]:
# Train risk prediction model
print("Training Climate Risk Model...")
risk_metrics = model.train_risk_model(X, y_risk)

print("\nRisk Model Performance:")
print(f"RMSE: {risk_metrics['rmse']:.4f}")
print(f"Cross-Validation RMSE: {risk_metrics['cv_rmse']:.4f}")

# Plot feature importance
visualizer.plot_feature_importance(risk_metrics['feature_importance'], 'Climate Risk Model', save_plot=False)

In [ ]:
# Train yield prediction model
print("Training Crop Yield Model...")
yield_metrics = model.train_yield_model(X, y_yield)

print("\nYield Model Performance:")
print(f"RMSE: {yield_metrics['rmse']:.4f}")
print(f"Cross-Validation RMSE: {yield_metrics['cv_rmse']:.4f}")

# Plot feature importance
visualizer.plot_feature_importance(yield_metrics['feature_importance'], 'Crop Yield Model', save_plot=False)

## 5. Model Predictions and Crop Recommendations

In [ ]:
# Sample prediction
sample_data = {
    'temperature_avg': 28.5,
    'temperature_max': 35.2,
    'temperature_min': 22.1,
    'rainfall': 450.0,
    'humidity': 68.5,
    'wind_speed': 12.3,
    'solar_radiation': 220.5,
    'region_encoded': 2,
    'crop_type_encoded': 1
}

# Create sample DataFrame
sample_df = pd.DataFrame([sample_data])

# Add engineered features
sample_df['temp_stress'] = np.where(sample_df['temperature_max'] > 35, 1, 0)
sample_df['cold_stress'] = np.where(sample_df['temperature_min'] < 5, 1, 0)
sample_df['drought_risk'] = np.where(sample_df['rainfall'] < 200, 1, 0)
sample_df['flood_risk'] = np.where(sample_df['rainfall'] > 1000, 1, 0)
sample_df['temp_range'] = sample_df['temperature_max'] - sample_df['temperature_min']
sample_df['growing_degree_days'] = np.maximum(0, sample_df['temperature_avg'] - 10)
sample_df['humidity_stress'] = np.where((sample_df['humidity'] < 30) | (sample_df['humidity'] > 90), 1, 0)

# Make predictions
risk_score = model.predict_risk_score(sample_df)[0]
predicted_yield = model.predict_yield(sample_df)[0]
risk_category = model.get_risk_category([risk_score])[0]

print("Sample Climate Data:")
for key, value in sample_data.items():
    print(f"{key}: {value}")

print(f"\nPredictions:")
print(f"Climate Risk Score: {risk_score:.3f}")
print(f"Risk Category: {risk_category}")
print(f"Predicted Yield: {predicted_yield:.2f} tons/hectare")

In [ ]:
# Crop recommendations
recommender = CropRecommender()

# Prepare climate data for recommender
climate_data_for_rec = {
    'temperature_avg': sample_data['temperature_avg'],
    'rainfall': sample_data['rainfall'],
    'humidity': sample_data['humidity'],
    'drought_risk': sample_df['drought_risk'].iloc[0],
    'temp_stress': sample_df['temp_stress'].iloc[0],
    'cold_stress': sample_df['cold_stress'].iloc[0],
    'flood_risk': sample_df['flood_risk'].iloc[0],
    'humidity_stress': sample_df['humidity_stress'].iloc[0]
}

# Generate recommendation report
recommendation_report = recommender.generate_recommendation_report(climate_data_for_rec)

print("\nCrop Recommendations:")
print("=" * 50)

print(f"\nClimate Summary:")
print(f"Overall Risk Score: {recommendation_report['climate_summary']['overall_risk_score']:.3f}")
print(f"Risk Category: {recommendation_report['climate_summary']['risk_category']}")

print(f"\nTop 5 Recommended Crops:")
for i, crop_rec in enumerate(recommendation_report['recommended_crops'][:5], 1):
    print(f"{i}. {crop_rec['crop'].title()}: {crop_rec['suitability_percentage']} suitable")

print(f"\nClimate Risks Detected:")
for risk, value in recommendation_report['climate_risks'].items():
    status = 'Yes' if value > 0 else 'No'
    print(f"- {risk.replace('_', ' ').title()}: {status}")

In [ ]:
# Adaptation strategies for top crop
top_crop = recommendation_report['recommended_crops'][0]['crop']
strategies = recommendation_report['adaptation_strategies'][top_crop]

print(f"\nAdaptation Strategies for {top_crop.title()}:")
print("=" * 50)
for i, strategy in enumerate(strategies, 1):
    print(f"{i}. {strategy}")

## 6. Interactive Dashboard

In [ ]:
# Create interactive dashboard
visualizer.create_interactive_dashboard(processed_df, save_plot=False)

## 7. Model Evaluation and Insights

In [ ]:
# Model performance summary
print("Model Performance Summary")
print("=" * 50)
print(f"\nClimate Risk Model:")
print(f"- RMSE: {risk_metrics['rmse']:.4f}")
print(f"- CV RMSE: {risk_metrics['cv_rmse']:.4f}")
print(f"- Model Type: XGBoost Regressor")

print(f"\nCrop Yield Model:")
print(f"- RMSE: {yield_metrics['rmse']:.4f}")
print(f"- CV RMSE: {yield_metrics['cv_rmse']:.4f}")
print(f"- Model Type: LightGBM Regressor")

# Key insights
print(f"\nKey Insights:")
print(f"- Dataset contains {processed_df.shape[0]} records across {processed_df['region'].nunique()} regions")
print(f"- {processed_df['crop_type'].nunique()} different crop types analyzed")
print(f"- Average climate risk score: {processed_df['climate_risk_score'].mean():.3f}")
print(f"- Average yield: {processed_df['yield_tons_per_hectare'].mean():.2f} tons/hectare")

# Risk distribution
risk_distribution = processed_df['climate_risk_score'].apply(
    lambda x: 'Low' if x < 0.3 else 'Medium' if x < 0.6 else 'High'
).value_counts()

print(f"\nRisk Distribution:")
for risk_level, count in risk_distribution.items():
    percentage = (count / len(processed_df)) * 100
    print(f"- {risk_level} Risk: {count} records ({percentage:.1f}%)")

## 8. Conclusions and Recommendations

### Model Performance
- The climate risk prediction model achieved good performance with cross-validation RMSE
- The crop yield prediction model shows reliable predictions for agricultural planning

### Key Findings
- Temperature and rainfall are the most important factors for crop yield prediction
- Climate stress indicators significantly impact crop suitability
- Regional variations play an important role in crop recommendations

### SDG Impact
- **SDG 2 (Zero Hunger)**: Helps farmers optimize crop selection and improve food security
- **SDG 13 (Climate Action)**: Provides climate adaptation strategies for sustainable agriculture

### Future Improvements
- Incorporate more detailed soil data
- Add seasonal climate patterns
- Include economic factors in crop recommendations
- Develop real-time prediction capabilities